# EGFR Atomistic Frustration Pipeline
**Chen et al. (2020) Nature Communications** - Independent Reimplementation  
61 EGFR-inhibitor complexes (51 unique ligands; expanded from the paper's 4)

---
**Run order:** Stage 0 -> 1 -> 2 (unit tests) -> 3 (validation) -> 4 (EGFR analysis)

## Stage 0 — Environment Setup & PyRosetta Verification

In [ ]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / "config.yaml").is_file():
    project_root = project_root.parent
if not (project_root / "config.yaml").is_file():
    raise FileNotFoundError("Could not find the repository config.yaml")
sys.path.insert(0, str(project_root / "src"))

# Verify all dependencies
import numpy as np
import pandas as pd
import scipy
import matplotlib
import yaml
import requests
from openbabel import openbabel as ob

with (project_root / "config.yaml").open() as config_file:
    cfg = yaml.safe_load(config_file)
cfg["paths"] = {
    "raw_pdb": str(project_root / "data" / "raw_pdb"),
    "processed": str(project_root / "data" / "processed"),
    "ligands": str(project_root / "data" / "ligands"),
    "params": str(project_root / "data" / "ligands" / "params"),
    "results": str(project_root / "results"),
    "checkpoints": str(project_root / "checkpoints"),
    "candidates_csv": str(project_root / "results" / "metadata" / "egfr_ligand_inventory.csv"),
    "prep_summary_csv": str(project_root / "results" / "preparation_summary.csv"),
    "ligand_status_csv": str(project_root / "results" / "metadata" / "ligand_parameterization_status.csv"),
}

print(f"Repository: {project_root}")
print(f"numpy {np.__version__}, pandas {pd.__version__}, scipy {scipy.__version__}")
print(f"OpenBabel {ob.OBReleaseVersion()}")

In [ ]:
import pyrosetta
pyrosetta.init("-mute all")
print("PyRosetta OK")

# Quick test: score lysozyme
pdb_path = Path(cfg["paths"]["raw_pdb"]) / "1LYZ.pdb"
if not pdb_path.exists():
    response = requests.get("https://files.rcsb.org/download/1LYZ.pdb", timeout=30)
    response.raise_for_status()
    pdb_path.parent.mkdir(parents=True, exist_ok=True)
    pdb_path.write_bytes(response.content)

pose = pyrosetta.pose_from_pdb(str(pdb_path))
sfxn = pyrosetta.create_score_function("ref2015")
score = sfxn(pose)
print(f"1LYZ: {pose.total_residue()} residues, REF2015 score = {score:.2f} REU")
print("Stage 0 PASSED")

## Stage 1 — Data Preparation

Download all 25 PDB structures, extract ligands, generate Rosetta .params files.

In [ ]:
# Show candidate list
df_cands = pd.read_csv(cfg["paths"]["candidates_csv"])
print(f"Candidates: {len(df_cands)} structures")
print(df_cands[["pdb_id", "ligand_comp_id", "affinity_pM", "resolution_A"]].to_string(index=False))

In [ ]:
prep = pd.read_csv(cfg["paths"]["prep_summary_csv"])
ligand_status = pd.read_csv(cfg["paths"]["ligand_status_csv"])

prepared = df_cands.merge(
    prep[["pdb_id", "status", "rosetta_ligand_comp_id"]].rename(
        columns={"status": "prep_status"}
    ),
    on="pdb_id",
    how="left",
).merge(
    ligand_status[["ligand_comp_id", "status"]].rename(
        columns={"status": "params_status"}
    ),
    on="ligand_comp_id",
    how="left",
)

params_dir = project_root / cfg["paths"]["params"]
prepared["params_file"] = prepared["rosetta_ligand_comp_id"].map(
    lambda ligand: params_dir / f"{ligand}.params" if pd.notna(ligand) else None
)
prepared["params_file_exists"] = prepared["params_file"].map(
    lambda path: path.exists() if path is not None else False
)
prepared["ready_for_analysis"] = (
    prepared["prep_status"].eq("OK")
    & prepared["params_status"].eq("OK")
    & prepared["params_file_exists"]
)

print(f"Ready for analysis: {prepared['ready_for_analysis'].sum()}/{len(prepared)} structures")
print(
    prepared.loc[
        ~prepared["ready_for_analysis"],
        ["pdb_id", "ligand_comp_id", "prep_status", "params_status", "params_file_exists"],
    ].to_string(index=False)
)

In [ ]:
# Verify poses load correctly
from run_pipeline import load_pose_with_ligand, find_ligand_resnum

df_ready = prepared[prepared["ready_for_analysis"]].copy()
processed_dir = Path(cfg["paths"]["processed"])
params_dir = Path(cfg["paths"]["params"])
summaries = []
sfxn = pyrosetta.create_score_function("ref2015")

for _, row in df_ready.iterrows():
    pdb_id = row["pdb_id"]
    rosetta_ligand_comp_id = row["rosetta_ligand_comp_id"]
    pdb_file = processed_dir / f"{pdb_id}_clean.pdb"
    params_file = params_dir / f"{rosetta_ligand_comp_id}.params"
    if not pdb_file.exists() or not params_file.exists():
        summaries.append({"pdb_id": pdb_id, "status": "missing"})
        continue
    try:
        pose = load_pose_with_ligand(str(pdb_file), str(params_file))
        score = sfxn(pose)
        ligand_resnum = find_ligand_resnum(pose, rosetta_ligand_comp_id)
        summaries.append({
            "pdb_id": pdb_id,
            "n_residues": pose.total_residue(),
            "lig_resnum": ligand_resnum,
            "score_REU": round(score, 2),
            "status": "ok",
        })
        print(f"  {pdb_id}: {pose.total_residue()} res, lig@{ligand_resnum}, score={score:.1f}")
    except Exception as error:
        summaries.append({"pdb_id": pdb_id, "status": f"error: {error}"})
        print(f"  {pdb_id} ERROR: {error}")

df_poses = pd.DataFrame(summaries)
print(f"\nSuccessfully loaded: {(df_poses['status'] == 'ok').sum()}/{len(df_poses)}")
print("Stage 1 PASSED" if df_poses["status"].eq("ok").all() else "Stage 1 WARNING: review failed structures")

## Stage 2 — Unit Tests

Verify Eq. 1, Eq. 2 implementation on a small protein before running full analysis.

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "pytest", "src/test_frustration.py", "-v", "--tb=short"],
    capture_output=True,
    text=True,
    cwd=project_root,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

In [ ]:
# Quick manual sanity check: pairwise energy symmetry
from frustration import get_protein_contacts, pairwise_energy, native_aa_frequency

pose_test = pyrosetta.pose_from_pdb(str(Path(cfg["paths"]["raw_pdb"]) / "1LYZ.pdb"))
sfxn(pose_test)
e_59 = pairwise_energy(pose_test, sfxn, 5, 9, exclude_fa_rep=True)
e_95 = pairwise_energy(pose_test, sfxn, 9, 5, exclude_fa_rep=True)
print(f"e(5,9) = {e_59:.6f}")
print(f"e(9,5) = {e_95:.6f}")
print(f"Symmetry OK: {abs(e_59 - e_95) < 1e-6}")

aa_freq = native_aa_frequency(pose_test)
print(f"\nAmino acid frequency (sum={sum(aa_freq.values()):.6f}):")
print({amino_acid: f"{frequency:.3f}" for amino_acid, frequency in sorted(aa_freq.items())})
print("Stage 2 PASSED")

## Stage 3 — Validation on Lysozyme (1LYZ)

Expected: buried core contacts → mostly minimally frustrated  
Surface contacts → more neutral/highly frustrated

In [ ]:
import importlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import frustration
import run_pipeline

importlib.reload(frustration)
importlib.reload(run_pipeline)
from run_pipeline import run_validation

df_lyz = run_validation(cfg, n_decoys=cfg["frustration"]["n_decoys"])

print(f"\nLysozyme frustration results (n_decoys={cfg['frustration']['n_decoys']}):")
print(df_lyz["frustration_class"].value_counts())
print("\nFrustration index stats:")
print(df_lyz["F_index"].describe().round(3))

In [ ]:
# Display validation figure when Stage 3 has generated it
from IPython.display import Image, display

validation_image = Path(cfg["paths"]["results"]) / "validation_lysozyme.png"
if validation_image.exists():
    display(Image(filename=str(validation_image)))
else:
    print(f"Validation figure not found: {validation_image}")

# Core/surface check via SASA
from Bio.PDB import PDBParser
from Bio.PDB.SASA import ShrakeRupley

parser = PDBParser(QUIET=True)
structure = parser.get_structure("lyz", str(Path(cfg["paths"]["raw_pdb"]) / "1LYZ.pdb"))
shrake_rupley = ShrakeRupley()
shrake_rupley.compute(structure, level="R")

sasa_by_seqid = {residue.id[1]: residue.sasa for residue in structure.get_residues()}

sasa_buried_threshold = 20
core_contacts = []
surface_contacts = []
for _, contact in df_lyz.iterrows():
    sasa_i = sasa_by_seqid.get(contact["resi"], 999)
    sasa_j = sasa_by_seqid.get(contact["resj"], 999)
    if (sasa_i + sasa_j) / 2 < sasa_buried_threshold:
        core_contacts.append(contact["frustration_class"])
    else:
        surface_contacts.append(contact["frustration_class"])

core_min_fraction = core_contacts.count("minimally_frustrated") / len(core_contacts) if core_contacts else 0
surface_min_fraction = surface_contacts.count("minimally_frustrated") / len(surface_contacts) if surface_contacts else 0

print(f"Core contacts (SASA < {sasa_buried_threshold} A^2): {len(core_contacts)} total")
print(f"  {100 * core_min_fraction:.0f}% minimally frustrated")
print(f"Surface contacts: {len(surface_contacts)} total")
print(f"  {100 * surface_min_fraction:.0f}% minimally frustrated")
print(f"\nCore > surface: {core_min_fraction > surface_min_fraction}")
print("Stage 3 PASSED" if core_min_fraction > surface_min_fraction else "Stage 3 WARNING: unexpected pattern, review code")

## Stage 4 — EGFR Analysis

Run frustration survey on all 61 EGFR-inhibitor complexes (51 unique ligands) and correlate minimally frustrated ligand-pocket contacts with binding affinity.

In [ ]:
# --- 4a. EGFR survey controls ---
# "single" parallelizes decoys; "all" parallelizes structures.
importlib.reload(frustration)
importlib.reload(run_pipeline)
from run_pipeline import (
    default_worker_count,
    load_candidates,
    run_all_egfr,
    run_single_structure,
    with_output_directories,
)

analysis_mode = "all"  # "single" or "all"
single_pdb_id = "3W2O"  # Used only when analysis_mode == "single"
n_decoys = 50
n_jobs = default_worker_count()  # Uses all available logical CPUs by default.
pdb_ids = None  # All-mode smoke test example: {"1XKK", "5GMP"}
results_dir_override = None  # Example: "runs/method-a/results"
checkpoints_dir_override = None  # Example: "runs/method-a/checkpoints"

def output_path(value):
    if value is None:
        return None
    path = Path(value)
    return str(path if path.is_absolute() else project_root / path)

analysis_cfg = with_output_directories(
    cfg,
    results_dir=output_path(results_dir_override),
    checkpoints_dir=output_path(checkpoints_dir_override),
)
print(f"Worker count: {n_jobs}")
print(f"Results directory: {analysis_cfg['paths']['results']}")
print(f"Checkpoints directory: {analysis_cfg['paths']['checkpoints']}")

if analysis_mode == "single":
    candidates = load_candidates(analysis_cfg)
    selected = candidates[candidates["pdb_id"] == single_pdb_id]
    if len(selected) != 1:
        raise ValueError(f"Expected one ready candidate for {single_pdb_id}, found {len(selected)}")

    candidate = selected.iloc[0]
    scorefxn = pyrosetta.create_score_function(analysis_cfg["energy"]["score_function"])
    summary = run_single_structure(
        pdb_id=candidate.pdb_id,
        ligand_comp_id=candidate.ligand_comp_id,
        rosetta_ligand_comp_id=candidate.rosetta_ligand_comp_id,
        cfg=analysis_cfg,
        n_decoys=n_decoys,
        n_jobs_decoys=n_jobs,
        scorefxn=scorefxn,
        seed=analysis_cfg["frustration"]["seed"],
    )
    if summary is None:
        raise RuntimeError(f"Analysis failed for {single_pdb_id}")

    df_results = pd.DataFrame([{
        "pdb_id": candidate.pdb_id,
        "ligand_comp_id": candidate.ligand_comp_id,
        "affinity_pM": candidate.affinity_pM,
        "log10_affinity_pM": np.log10(candidate.affinity_pM),
        **summary,
    }])
else:
    if analysis_mode != "all":
        raise ValueError("analysis_mode must be 'single' or 'all'")
    df_results = run_all_egfr(
        analysis_cfg,
        n_decoys=n_decoys,
        n_jobs=n_jobs,
        pdb_ids=pdb_ids,
    )

print(
    df_results[
        ["pdb_id", "ligand_comp_id", "affinity_pM", "n_minimally_frustrated", "n_contacts_total"]
    ]
)

Worker count: 4
Results directory: /workspaces/egfr_analysis_pipeline_withRosetta/results
Checkpoints directory: /workspaces/egfr_analysis_pipeline_withRosetta/checkpoints
2026-08-09 21:00:29,362 [INFO] 3W2O: Loading pose...


2026-08-09 21:00:29,621 [INFO]   Total residues: 306
2026-08-09 21:00:29,777 [INFO]   1751 protein-protein contacts
2026-08-09 21:00:29,784 [INFO]   62 ligand-protein contact residues
2026-08-09 21:00:29,785 [INFO] Computing native energies...
2026-08-09 21:03:13,658 [INFO] Generating 50 decoys with 4 spawned workers.


/opt/conda/envs/frustrato/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Parallel decoys:   0%|          | 0/50 [00:00<?, ?decoy/s]

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python310.ubuntu 2026.30+release.bc091c65b862304d5a2a1bed5f4648508439ab55 2026-07-23T17:32:39] retrieved from: http://www.pyrosetta.org
┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │

Parallel decoys:  46%|████▌     | 23/50 [28:39<27:06, 60.25s/decoy]  

In [ ]:
# --- 4b. Display correlation figure ---
correlation_image = Path(analysis_cfg["paths"]["results"]) / "egfr_correlation.png"
if correlation_image.exists():
    img = mpimg.imread(correlation_image)
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.imshow(img)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print(f"Correlation plot not found: {correlation_image}")

In [ ]:
# --- 4c. Correlation analysis: full set ---
from scipy import stats

if len(df_results) >= 3:
    n_minimally_frustrated = df_results["n_minimally_frustrated"].to_numpy()
    log10_affinity = df_results["log10_affinity_pM"].to_numpy()
    correlation, p_value = stats.pearsonr(n_minimally_frustrated, log10_affinity)
    print(f"Full set (n={len(df_results)}) Pearson r = {correlation:.3f}, p = {p_value:.3f}")
else:
    print(f"Insufficient data for correlation (n={len(df_results)} < 3)")

print("\nStage 4 COMPLETE")

In [ ]:
# --- 4d. Save results README ---
results_dir = Path(analysis_cfg["paths"]["results"])
readme = f"""# EGFR Frustration Analysis Results

## Parameters
- n_decoys: {n_decoys}
- protein-protein contact cutoff: {analysis_cfg['contacts']['protein_protein_cutoff_A']} A
- ligand-protein contact cutoff: {analysis_cfg['contacts']['ligand_protein_cutoff_A']} A
- seed: {analysis_cfg['frustration']['seed']}
- exclude_fa_rep: {analysis_cfg['frustration']['exclude_fa_rep']}

## Dataset
- Structures analyzed: {len(df_results)}
- Unique ligands: {df_results['ligand_comp_id'].nunique()}
- Affinity range: {df_results['affinity_pM'].min():.3g} to {df_results['affinity_pM'].max():.3g} pM

## Key files
- egfr_frustration_summary.csv: per-structure results
- egfr_correlation.png: minimally frustrated contacts vs log10(affinity)
- validation_lysozyme.png: Lysozyme validation
"""
results_dir.mkdir(parents=True, exist_ok=True)
(results_dir / "README.md").write_text(readme)
print(f"{results_dir / 'README.md'} written")